# SynBioCrow 2.3 — Freeze Release\nCreates a reproducibility/freeze manifest hashing the sealed predictions, scored benchmark, error analysis, similarity policy, and manuscript at the current develop/2.3 commit.\n

In [ ]:
import sys,subprocess,shutil,json,hashlib,zipfile,time
from pathlib import Path
from google.colab import drive, files
drive.mount("/content/drive",force_remount=False)
REPO=Path("/content/SynBioCrow_postbench")
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(["git","clone","--depth","1","--branch","develop/2.3",
                "https://github.com/theiman112860/SynBioCrow.git",str(REPO)],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-e",str(REPO)],check=True)

SEALED=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/galaxy_benchmark_sealed")
SCORED=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/galaxy_benchmark_scored/galaxy_benchmark_evaluation.json")
ERRORS=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/galaxy_benchmark_error_analysis/galaxy_benchmark_error_analysis.json")
OUT=Path("/content/drive/MyDrive/SynBioCrow/release/2.3_freeze")
OUT.mkdir(parents=True,exist_ok=True)
SCRIPT=REPO/"scripts/freeze_release_2_3.py"
subprocess.run([sys.executable,"-m","py_compile",str(SCRIPT)],check=True)
subprocess.run([sys.executable,str(SCRIPT),
                "--repo-root",str(REPO),
                "--sealed-dir",str(SEALED),
                "--scored",str(SCORED),
                "--error-analysis",str(ERRORS),
                "--output-dir",str(OUT)],cwd=REPO,check=True)
bundle=OUT/"SynBioCrow_2_3_FREEZE_PACKAGE.zip"
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.glob("*")):
        if p.is_file() and p!=bundle: z.write(p,p.name)
print("BUNDLE",bundle)
print("SHA256",hashlib.sha256(bundle.read_bytes()).hexdigest())
files.download(str(bundle))
